## Future idea: PySpark processing on the full dataset
PySpark uses lazy evaluation by default for all data transformations.
Purpose:
- filter observations for European weather stations,
- select relevant weather metrics,
- perform yearly/monthly/station-level aggregations,
- compare PySpark transformations with Pandas and Polars workflows,
- practice Spark DataFrame syntax for scalable data processing.

- implement a similar weather processing workflow in PySpark
- practice Spark DataFrame transformations
- work with lazy execution and query plans
- perform filtering, joins and aggregations on the full dataset

This notebook should demonstrate how the same data processing logic can be expressed in PySpark and how Spark can be used for larger datasets or distributed processing scenarios.

In [ ]:
# Import libraries
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pathlib import Path

# Initalize the Spark Session which is 
# the main entry point for working with Spark
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('Weather Processing')
    .getOrCreate()
)

WEATHER = Path(
    r"C:\Users\zychl\Desktop\Data Engineering\weather_data_processing\00_raw_data\weather.parquet"
)

STATIONS = Path(
    r"C:\Users\zychl\Desktop\Data Engineering\weather_data_processing\00_raw_data\stations.csv"
)

# Read weather.parquet and stations.csv files
# Convert the Path object to a string because Spark expects the file path as text
df_raw_weather = (
    spark.read
    .parquet(str(WEATHER))
    )

df_raw_stations = (
    spark.read
    .option('header', True)
    .csv(str(STATIONS))
    )

# Show DataFrames
{
    'weather': df_raw_weather.show(5), 
    'stations': df_raw_stations.show(5)
}

+-----------+----------------+------+-----+----------------+------------+-----------+----------------+
|    station|observation_date|metric|value|measurement_flag|quality_flag|source_flag|observation_time|
+-----------+----------------+------+-----+----------------+------------+-----------+----------------+
|ASN00012038|        20150101|  TMAX|  353|            NULL|        NULL|          a|            NULL|
|ASN00012038|        20150101|  TMIN|  163|            NULL|        NULL|          a|            NULL|
|ASN00012038|        20150101|  PRCP|    0|            NULL|        NULL|          a|            NULL|
|ASN00012038|        20150101|  TAVG|  272|               H|        NULL|          S|            NULL|
|ASN00012043|        20150101|  PRCP|    0|            NULL|        NULL|          a|            NULL|
+-----------+----------------+------+-----+----------------+------------+-----------+----------------+
only showing top 5 rows
+-----------+--------+---------+---------+-----+-

{'weather': None, 'stations': None}

In [27]:
# Inspect DataFrame schemas
df_raw_weather.printSchema()
df_raw_stations.printSchema()

root
 |-- station: string (nullable = true)
 |-- observation_date: long (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: long (nullable = true)
 |-- measurement_flag: string (nullable = true)
 |-- quality_flag: string (nullable = true)
 |-- source_flag: string (nullable = true)
 |-- observation_time: double (nullable = true)

root
 |-- station: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- elevation: string (nullable = true)
 |-- state: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- gsn_flag: string (nullable = true)
 |-- hcn_flag: string (nullable = true)
 |-- wmo_id: string (nullable = true)



| Spark type | Polars type |
|---|---|
| `string` | `String` |
| `long` | `Int64` |
| `double` | `Float64` |
| `boolean` | `Boolean` |
| `date` | `Date` |
| `timestamp` | `Datetime` |

In [ ]:
from pyspark.sql import types as T

schema = T.StructType([
T.StructField("transaction_id", T.IntegerType(), False),
T.StructField("customer_name", T.StringType(), False),
T.StructField("net_amount", T.DoubleType(), True),
T.StructField("tax_amount", T.DoubleType(), True),
T.StructField("is_member", T.BooleanType(), True),
])